In [ ]:

# JARVIS BRAIN v1 - Qwen3-4B-Base baseline (deterministic 108-item suite)
import importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.50,<5", "accelerate>=1.2", "safetensors", "psutil"])

from pathlib import Path as _P

eval_dir_candidates = [
    _P("/kaggle/input/jarvis-brain-v1-eval"),
    _P("/kaggle/input/datasets/videlisndichi/jarvis-brain-v1-eval"),
]
eval_dir = next((c for c in eval_dir_candidates if (c / "evaluator.py").exists()), None)
if eval_dir is None:
    for hit in sorted(_P("/kaggle/input").rglob("evaluator.py")):
        if "jarvis-brain-v1-eval" in str(hit.parent):
            eval_dir = hit.parent
            break
assert eval_dir is not None, "eval dataset not mounted (evaluator.py missing)"

work = Path("/kaggle/working"); work.mkdir(exist_ok=True)

import torch
torch.manual_seed(0); torch.random.manual_seed(0)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")

spec = importlib.util.spec_from_file_location("jarvis_eval", eval_dir / "evaluator.py")
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)

print("IDENTITY", json.dumps({"base_model": ev.BASE_MODEL,
                              "base_revision": ev.BASE_REVISION, "adapter": None}))

rep = ev.run_eval(mode="base", out_dir=str(work), suite_dir=str(eval_dir))

shutil.copyfile(work / "eval.json", work / "qwen3_4b_baseline.json")
shutil.copyfile(work / "eval.md", work / "qwen3_4b_baseline.md")

print("BASELINE_DONE", json.dumps({
    "identity": rep["identity"],
    "hardware": rep["hardware"],
    "overall_score": rep["overall_score"],
    "items_scored": rep["items_scored"],
    "items_correct": rep["items_correct"],
    "families": rep["families"]}))
